# 作业 5.1：液体闪烁体的脉冲形状甄别

本作业使用 $2\times2$ 英寸 BC501A 液体闪烁体探测器测得的钚碳中子源数据。信号由 XIA 500 MS/s、14 bit 数字化采集卡记录，因此相邻采样点间隔为 2 ns。

数据文件 [liquidpsd_tree.root](liquidpsd_tree.root) 中的 TTree `wave` 包含 10 000 个 entries，每个 entry 对应一次触发，branch `adc[250]` 保存该次触发的 250 个 ADC samples。波形保留了基线偏置和触发位置的变化，需要在积分前完成基线修正和时间对齐。


<div class="code-language-switch" role="group" aria-label="Code language">
  <span>Code language:</span>
  <button type="button" data-code-language="python" aria-pressed="true">Python / PyROOT</button>
  <button type="button" data-code-language="cpp" aria-pressed="false">ROOT C++</button>
</div>

<style>
.code-language-switch { display:none; gap:.5rem; align-items:center; margin:1rem 0; }
.code-language-switch button { padding:.3rem .8rem; border:1px solid #b8b8b8; border-radius:4px; background:#fff; cursor:pointer; }
.code-language-switch button[aria-pressed="true"] { color:#fff; background:#2f6f9f; border-color:#2f6f9f; }
.pyroot-code-marker { display:none; }
.pyroot-code-marker + .highlight { margin:.5rem 0 1rem; border:1px solid #d5d5d5; background:#f7f7f7; }
.pyroot-code-marker + .highlight pre { margin:0; padding:.75rem 1rem; overflow-x:auto; }
.pyroot-code-cell[hidden], .cpp-code-cell[hidden] { display:none !important; }
</style>

<script>
document.addEventListener("DOMContentLoaded", function () {
  const buttons = document.querySelectorAll(".code-language-switch button");
  const pythonCells = Array.from(document.querySelectorAll(".pyroot-code-marker"))
    .map(function (marker) { return marker.closest(".jp-MarkdownCell"); })
    .filter(Boolean);
  pythonCells.forEach(function (cell) { cell.classList.add("pyroot-code-cell"); });
  const cppCells = Array.from(document.querySelectorAll(".jp-CodeCell"));
  cppCells.forEach(function (cell) { cell.classList.add("cpp-code-cell"); });

  function selectLanguage(language) {
    pythonCells.forEach(function (cell) { cell.hidden = language !== "python"; });
    cppCells.forEach(function (cell) { cell.hidden = language !== "cpp"; });
    buttons.forEach(function (button) {
      button.setAttribute("aria-pressed", String(button.dataset.codeLanguage === language));
    });
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () { selectLanguage(button.dataset.codeLanguage); });
  });
  document.querySelector(".code-language-switch").style.display = "flex";
  selectLanguage("python");
});
</script>


## 方法

液体有机闪烁体中，neutron 主要通过反冲质子沉积能量，gamma ray 主要通过电子沉积能量。两类带电粒子产生的快、慢发光成分比例不同；在相近的脉冲总积分下，neutron pulse 通常具有更大的慢成分。

用触发前 $N_b$ 个采样点估计每条波形的基线

$$b=\frac{1}{N_b}\sum_{i=0}^{N_b-1}y_i,\qquad y_i'=y_i-b.$$

同一类脉冲在记录窗口中的触发位置会有少量变化。如果直接使用固定积分门，门的边界会落在脉冲的不同位置，使 $Q_{\rm fast}$ 和 PSD 产生额外展宽。因此应在积分前把波形对齐。

本数据中的脉冲峰顶尖锐，最大采样点能够稳定地标记脉冲位置，因此这里用 maximum 所在的采样点 $k_{\max}$ 对齐。若峰顶较宽或峰顶噪声明显，maximum 的采样点容易跳动，这种方法便不再可靠；实际分析中应改用前沿的 fast filter 或 constant-fraction 定时。简单的固定阈值 leading-edge 还会随脉冲幅度产生 time walk。

取第一条波形的 maximum 位置作为参考位置 $k_0$。对其余波形令 $\Delta=k_0-k_{\max}$，对齐后的数组满足

$$\widetilde y_j=y'_{j-\Delta}.$$

也就是说，写入输出位置 $j$ 时，应读取原数组位置 $j-\Delta$。时间平移只改变采样值所在的位置；移出记录窗口的点丢弃，空出的点补零。

对齐后选取 $t_0<t_1<t_2$，定义

$$Q_{\rm fast}=\sum_{i=t_0}^{t_1-1}y_i',\qquad Q_{\rm total}=\sum_{i=t_0}^{t_2-1}y_i',$$

$$Q_{\rm tail}=Q_{\rm total}-Q_{\rm fast},\qquad PSD=\frac{Q_{\rm tail}}{Q_{\rm total}}.$$

$t_0$ 应位于脉冲上升沿之前，$t_2$ 应覆盖脉冲回到基线的过程；$t_1$ 决定快成分与慢成分的分界。采样点序号乘以 2 ns 即为相对时间。

PSD 随脉冲幅度变化，因此 FoM 应在固定的 $Q_{\rm total}$ 区间内比较。把该区间内的 PSD 投影为一维分布，分别拟合 gamma 与 neutron peak，定义

$$FoM=\frac{|\mu_n-\mu_\gamma|}{FWHM_n+FWHM_\gamma}.$$

FoM 越大表示两类事例在所选能量区间内分离越好。积分门变化时，应保持同一 $Q_{\rm total}$ 区间和相同的拟合方法。


## 实例代码

### 读取 ROOT 文件

Get 读取名为 `wave` 的 TTree，GetEntries 返回波形数；每次 GetEntry 后，`adc` branch 给出当前波形的 250 个采样值。


<div class="pyroot-code-marker"></div>

~~~python
import math
from array import array
import ROOT

%jsroot on
ROOT.gStyle.SetOptStat(0)

input_file = ROOT.TFile.Open("liquidpsd_tree.root", "READ")
if not input_file or input_file.IsZombie():
    raise OSError("cannot open liquidpsd_tree.root")

wave = input_file.Get("wave")
pulse_count = wave.GetEntries()
sample_count = 250

def read_pulse(index):
    wave.GetEntry(index)
    return [float(wave.adc[i]) for i in range(sample_count)]

reference_pulse = read_pulse(0)
alignment_target = max(range(sample_count), key=reference_pulse.__getitem__)
print(f"{pulse_count} waveforms, {sample_count} samples per waveform; "
      f"reference maximum = {alignment_target}")
~~~


In [ ]:
//%jsroot on
#include "TCanvas.h"
#include "TBox.h"
#include "TFile.h"
#include "TF1.h"
#include "TFitResultPtr.h"
#include "TGraph.h"
#include "TH1D.h"
#include "TH2D.h"
#include "TLatex.h"
#include "TLegend.h"
#include "TLine.h"
#include "TMultiGraph.h"
#include "TPad.h"
#include "TStyle.h"
#include "TString.h"
#include "TTree.h"
#include <algorithm>
#include <cmath>
#include <iomanip>
#include <iostream>
#include <limits>
#include <numeric>
#include <stdexcept>
#include <utility>
#include <vector>

gStyle->SetOptStat(0);

auto inputFile = TFile::Open("liquidpsd_tree.root", "READ");
if (!inputFile || inputFile->IsZombie()) {
    throw std::runtime_error("cannot open liquidpsd_tree.root");
}
auto wave = dynamic_cast<TTree*>(inputFile->Get("wave"));
const int pulseCount = wave->GetEntries();
const int sampleCount = 250;
double adc[sampleCount];
wave->SetBranchAddress("adc", adc);
auto readPulse = [&](int index) {
    wave->GetEntry(index);
    return std::vector<double>(adc, adc + sampleCount);
};
auto referencePulse = readPulse(0);
const int alignmentTarget = std::distance(
    referencePulse.begin(),
    std::max_element(referencePulse.begin(), referencePulse.end()));
std::cout << pulseCount << " waveforms, " << sampleCount
          << " samples per waveform; reference maximum = "
          << alignmentTarget << std::endl;


### 观察原始波形

下面只画两条波形。原始基线约为 ADC 1660，pulse maximum 的采样点也并不完全相同。


<div class="pyroot-code-marker"></div>

~~~python
raw20 = read_pulse(20)
raw30 = read_pulse(30)
samples = array("d", range(sample_count))
raw_graph20 = ROOT.TGraph(sample_count, samples, array("d", raw20))
raw_graph30 = ROOT.TGraph(sample_count, samples, array("d", raw30))
raw_graph20.SetLineColor(ROOT.kBlue + 1)
raw_graph30.SetLineColor(ROOT.kRed + 1)

raw_graphs = ROOT.TMultiGraph()
raw_graphs.SetTitle("Raw waveforms;sample index;ADC value")
raw_graphs.Add(raw_graph20)
raw_graphs.Add(raw_graph30)
c_raw = ROOT.TCanvas("c_raw_py", "raw waveforms", 800, 480)
raw_graphs.Draw("AL")
c_raw.Draw()
c_raw.SaveAs("raw_waveforms.png")
~~~


In [ ]:
auto raw20 = readPulse(20);
auto raw30 = readPulse(30);
std::vector<double> samples(sampleCount);
std::iota(samples.begin(), samples.end(), 0.0);
auto rawGraph20 = new TGraph(sampleCount, samples.data(), raw20.data());
auto rawGraph30 = new TGraph(sampleCount, samples.data(), raw30.data());
rawGraph20->SetLineColor(kBlue + 1);
rawGraph30->SetLineColor(kRed + 1);

auto rawGraphs = new TMultiGraph();
rawGraphs->SetTitle("Raw waveforms;sample index;ADC value");
rawGraphs->Add(rawGraph20);
rawGraphs->Add(rawGraph30);
auto cRaw = new TCanvas("cRaw", "raw waveforms", 800, 480);
rawGraphs->Draw("AL");
cRaw->Draw();
cRaw->SaveAs("raw_waveforms.png");


<img src="raw_waveforms.png" alt="two raw BC501A waveforms" style="max-width:58%;" />


### 基线修正与时间对齐

示例使用前 40 个采样点估计基线，并把每条波形的 pulse maximum 对齐到第一条波形的 maximum 位置。图中先比较基线修正后但尚未对齐的波形，再显示对齐结果。代码中的 `source = output - shift` 对应上面的 $j-\Delta$；边界判断检查的也是实际读取的 source。


<div class="pyroot-code-marker"></div>

~~~python
def process_pulse(values, target, baseline_samples=40):
    baseline = sum(values[:baseline_samples]) / baseline_samples
    corrected = [value - baseline for value in values]
    peak_index = max(range(len(corrected)), key=corrected.__getitem__)
    shift = target - peak_index
    aligned = [0.0] * len(corrected)
    for output in range(len(corrected)):
        source = output - shift
        if 0 <= source < len(corrected):
            aligned[output] = corrected[source]
    return aligned, baseline, peak_index

pulse20, base20, peak20 = process_pulse(raw20, alignment_target)
pulse30, base30, peak30 = process_pulse(raw30, alignment_target)
corrected20 = [value - base20 for value in raw20]
corrected30 = [value - base30 for value in raw30]
before20 = ROOT.TGraph(sample_count, samples, array("d", corrected20))
before30 = ROOT.TGraph(sample_count, samples, array("d", corrected30))
after20 = ROOT.TGraph(sample_count, samples, array("d", pulse20))
after30 = ROOT.TGraph(sample_count, samples, array("d", pulse30))
for blue, red in ((before20, before30), (after20, after30)):
    blue.SetLineColor(ROOT.kBlue + 1)
    red.SetLineColor(ROOT.kRed + 1)

c_alignment = ROOT.TCanvas("c_alignment_py", "time alignment", 1000, 430)
c_alignment.Divide(2, 1)
c_alignment.cd(1)
before_graphs = ROOT.TMultiGraph()
before_graphs.SetTitle("Before alignment;sample index;ADC-baseline")
before_graphs.Add(before20)
before_graphs.Add(before30)
before_graphs.Draw("AL")
before_legend = ROOT.TLegend(0.48, 0.70, 0.88, 0.86)
before_legend.AddEntry(before20, f"event 20: kmax={peak20}", "l")
before_legend.AddEntry(before30, f"event 30: kmax={peak30}", "l")
before_legend.Draw()
c_alignment.cd(2)
after_graphs = ROOT.TMultiGraph()
after_graphs.SetTitle("After alignment;sample index;ADC-baseline")
after_graphs.Add(after20)
after_graphs.Add(after30)
after_graphs.Draw("AL")
target_line = ROOT.TLine(alignment_target, 0, alignment_target,
                         1.05 * max(max(pulse20), max(pulse30)))
target_line.SetLineStyle(2)
target_line.Draw()
target_label = ROOT.TLatex()
target_label.SetTextSize(0.035)
target_label.DrawLatex(alignment_target + 4,
                       0.92 * max(max(pulse20), max(pulse30)),
                       f"k0 = {alignment_target}")
c_alignment.Draw()
c_alignment.SaveAs("time_alignment.png")
print(f"baselines = {base20:.2f}, {base30:.2f}; "
      f"original maxima = {peak20}, {peak30}")
~~~


In [ ]:
struct ProcessedPulse {
    std::vector<double> values;
    double baseline;
    int peakIndex;
};

ProcessedPulse processPulse(const std::vector<double>& values, int target, int baselineSamples = 40) {
    const int n = values.size();
    std::vector<double> corrected(n);
    double baseline = 0.0;
    for (int i = 0; i < baselineSamples; ++i) baseline += values[i];
    baseline /= baselineSamples;
    for (int i = 0; i < n; ++i) corrected[i] = values[i] - baseline;

    const int peakIndex = std::distance(
        corrected.begin(), std::max_element(corrected.begin(), corrected.end()));
    const int shift = target - peakIndex;
    std::vector<double> aligned(n, 0.0);
    for (int output = 0; output < n; ++output) {
        const int source = output - shift;
        if (source >= 0 && source < n) aligned[output] = corrected[source];
    }
    return {aligned, baseline, peakIndex};
}

auto pulse20 = processPulse(raw20, alignmentTarget);
auto pulse30 = processPulse(raw30, alignmentTarget);
std::vector<double> corrected20(sampleCount);
std::vector<double> corrected30(sampleCount);
for (int i = 0; i < sampleCount; ++i) {
    corrected20[i] = raw20[i] - pulse20.baseline;
    corrected30[i] = raw30[i] - pulse30.baseline;
}
auto before20 = new TGraph(sampleCount, samples.data(), corrected20.data());
auto before30 = new TGraph(sampleCount, samples.data(), corrected30.data());
auto after20 = new TGraph(sampleCount, samples.data(), pulse20.values.data());
auto after30 = new TGraph(sampleCount, samples.data(), pulse30.values.data());
for (auto graph : {before20, after20}) graph->SetLineColor(kBlue + 1);
for (auto graph : {before30, after30}) graph->SetLineColor(kRed + 1);

auto cAlignment = new TCanvas("cAlignment", "time alignment", 1000, 430);
cAlignment->Divide(2, 1);
cAlignment->cd(1);
auto beforeGraphs = new TMultiGraph();
beforeGraphs->SetTitle("Before alignment;sample index;ADC-baseline");
beforeGraphs->Add(before20);
beforeGraphs->Add(before30);
beforeGraphs->Draw("AL");
auto beforeLegend = new TLegend(0.48, 0.70, 0.88, 0.86);
beforeLegend->AddEntry(before20, Form("event 20: kmax=%d", pulse20.peakIndex), "l");
beforeLegend->AddEntry(before30, Form("event 30: kmax=%d", pulse30.peakIndex), "l");
beforeLegend->Draw();
cAlignment->cd(2);
auto afterGraphs = new TMultiGraph();
afterGraphs->SetTitle("After alignment;sample index;ADC-baseline");
afterGraphs->Add(after20);
afterGraphs->Add(after30);
afterGraphs->Draw("AL");
const double alignedMaximum = std::max(
    *std::max_element(pulse20.values.begin(), pulse20.values.end()),
    *std::max_element(pulse30.values.begin(), pulse30.values.end()));
auto targetLine = new TLine(
    alignmentTarget, 0, alignmentTarget, 1.05 * alignedMaximum);
targetLine->SetLineStyle(2);
targetLine->Draw();
auto targetLabel = new TLatex();
targetLabel->SetTextSize(0.035);
targetLabel->DrawLatex(
    alignmentTarget + 4, 0.92 * alignedMaximum,
    Form("k0 = %d", alignmentTarget));
cAlignment->Draw();
cAlignment->SaveAs("time_alignment.png");
std::cout << std::fixed << std::setprecision(2)
          << "baselines = " << pulse20.baseline << ", " << pulse30.baseline
          << "; original maxima = " << pulse20.peakIndex << ", "
          << pulse30.peakIndex << std::endl;


<img src="time_alignment.png" alt="waveforms before and after time alignment" style="max-width:76%;" />

左图中两条波形的尖锐峰顶分别位于不同采样点；右图中两者都位于第一条波形给出的参考位置 $k_0=61$。这样，同一组积分门相对于每条脉冲具有相同位置。


### 积分门与脉冲的相对位置

fast gate 是 total gate 的前一部分；两者之差给出 tail integral。下面把示例积分门直接画在一条对齐后的脉冲上。


<div class="pyroot-code-marker"></div>

~~~python
gate_start, fast_end, total_end = 50, 72, 240
gate_pulse = [value / max(pulse20) for value in pulse20]
gate_graph = ROOT.TGraph(sample_count, samples, array("d", gate_pulse))
gate_graph.SetLineWidth(2)
gate_graph.SetTitle("Integration gates;sample index;normalized amplitude")

c_gate = ROOT.TCanvas("c_gate_py", "integration gates", 800, 480)
gate_graph.Draw("AL")
gate_graph.GetYaxis().SetRangeUser(-0.05, 1.18)
total_box = ROOT.TBox(gate_start, -0.05, total_end, 1.18)
total_box.SetFillColorAlpha(ROOT.kOrange - 3, 0.18)
total_box.SetLineColor(ROOT.kOrange + 7)
total_box.Draw("same")
fast_box = ROOT.TBox(gate_start, -0.05, fast_end, 1.18)
fast_box.SetFillColorAlpha(ROOT.kAzure - 9, 0.35)
fast_box.SetLineColor(ROOT.kAzure + 2)
fast_box.Draw("same")
gate_graph.Draw("L same")
gate_label = ROOT.TLatex()
gate_label.SetTextSize(0.035)
gate_label.DrawLatex(53, 0.90, "fast gate")
gate_label.DrawLatex(125, 1.07, "total gate")
c_gate.Draw()
c_gate.SaveAs("integration_gates.png")
~~~


In [ ]:
const int gateStart = 50;
const int fastEnd = 72;
const int totalEnd = 240;
std::vector<double> gatePulse = pulse20.values;
const double gateMaximum = *std::max_element(gatePulse.begin(), gatePulse.end());
for (double& value : gatePulse) value /= gateMaximum;
auto gateGraph = new TGraph(sampleCount, samples.data(), gatePulse.data());
gateGraph->SetLineWidth(2);
gateGraph->SetTitle("Integration gates;sample index;normalized amplitude");

auto cGate = new TCanvas("cGate", "integration gates", 800, 480);
gateGraph->Draw("AL");
gateGraph->GetYaxis()->SetRangeUser(-0.05, 1.18);
auto totalBox = new TBox(gateStart, -0.05, totalEnd, 1.18);
totalBox->SetFillColorAlpha(kOrange - 3, 0.18);
totalBox->SetLineColor(kOrange + 7);
totalBox->Draw("same");
auto fastBox = new TBox(gateStart, -0.05, fastEnd, 1.18);
fastBox->SetFillColorAlpha(kAzure - 9, 0.35);
fastBox->SetLineColor(kAzure + 2);
fastBox->Draw("same");
gateGraph->Draw("L same");
auto gateLabel = new TLatex();
gateLabel->SetTextSize(0.035);
gateLabel->DrawLatex(53, 0.90, "fast gate");
gateLabel->DrawLatex(125, 1.07, "total gate");
cGate->Draw();
cGate->SaveAs("integration_gates.png");


<img src="integration_gates.png" alt="fast and total integration gates on an aligned pulse" style="max-width:58%;" />


### 处理全部波形

下面保留每条对齐后的波形，后续改变积分门时不需要再次读取 ROOT 文件。代码同时统计 maximum 明显偏离主要触发位置的波形，但不据此自动删除事例。


<div class="pyroot-code-marker"></div>

~~~python
aligned_pulses = []
unusual_maxima = 0
for index in range(pulse_count):
    values, _, peak_index = process_pulse(read_pulse(index), alignment_target)
    aligned_pulses.append(values)
    if not 50 <= peak_index <= 70:
        unusual_maxima += 1
print(f"processed {len(aligned_pulses)} waveforms; "
      f"{unusual_maxima} maxima outside samples 50-70")
~~~


In [ ]:
std::vector<std::vector<double>> alignedPulses;
alignedPulses.reserve(pulseCount);
int unusualMaxima = 0;
for (int index = 0; index < pulseCount; ++index) {
    auto pulse = processPulse(readPulse(index), alignmentTarget);
    alignedPulses.push_back(std::move(pulse.values));
    if (pulse.peakIndex < 50 || pulse.peakIndex > 70) ++unusualMaxima;
}
std::cout << "processed " << alignedPulses.size() << " waveforms; "
          << unusualMaxima << " maxima outside samples 50-70" << std::endl;


### 电荷积分与 PSD 二维图

以下积分门只用于演示完整计算：fast gate 为 $[50,72)$，total gate 为 $[50,240)$，分别对应 44 ns 和 380 ns。学生应根据平均波形和 FoM 比较自己的积分门。


<div class="pyroot-code-marker"></div>

~~~python
total_charge = []
psd_value = []
h_tail_total = ROOT.TH2D(
    "h_tail_total_py", "Tail integral;Q_{total} (10^{3} ADC sum);Q_{tail} (10^{3} ADC sum)",
    250, 0, 150, 200, 0, 60
)
h_psd = ROOT.TH2D(
    "h_psd_py", "Pulse-shape discrimination;Q_{total} (10^{3} ADC sum);Q_{tail}/Q_{total}",
    250, 0, 150, 180, 0, 0.5
)

for values in aligned_pulses:
    q_fast = sum(values[gate_start:fast_end])
    q_total = sum(values[gate_start:total_end])
    q_tail = q_total - q_fast
    psd = q_tail / q_total if q_total > 0 else float("nan")
    total_charge.append(q_total)
    psd_value.append(psd)
    if math.isfinite(psd):
        h_tail_total.Fill(q_total / 1000.0, q_tail / 1000.0)
        h_psd.Fill(q_total / 1000.0, psd)

c_psd = ROOT.TCanvas("c_psd_py", "PSD maps", 1000, 430)
c_psd.Divide(2, 1)
c_psd.cd(1)
ROOT.gPad.SetLogz()
h_tail_total.Draw("colz")
c_psd.cd(2)
ROOT.gPad.SetLogz()
h_psd.Draw("colz")
c_psd.Draw()
c_psd.SaveAs("psd_maps.png")
~~~


In [ ]:
std::vector<double> totalCharge(pulseCount);
std::vector<double> psdValue(pulseCount, std::numeric_limits<double>::quiet_NaN());
auto hTailTotal = new TH2D(
    "hTailTotal", "Tail integral;Q_{total} (10^{3} ADC sum);Q_{tail} (10^{3} ADC sum)",
    250, 0, 150, 200, 0, 60);
auto hPsd = new TH2D(
    "hPsd", "Pulse-shape discrimination;Q_{total} (10^{3} ADC sum);Q_{tail}/Q_{total}",
    250, 0, 150, 180, 0, 0.5);

for (int index = 0; index < pulseCount; ++index) {
    const auto& values = alignedPulses[index];
    const double qFast = std::accumulate(
        values.begin() + gateStart, values.begin() + fastEnd, 0.0);
    const double qTotal = std::accumulate(
        values.begin() + gateStart, values.begin() + totalEnd, 0.0);
    const double qTail = qTotal - qFast;
    totalCharge[index] = qTotal;
    if (qTotal <= 0.0) continue;
    psdValue[index] = qTail / qTotal;
    hTailTotal->Fill(qTotal / 1000.0, qTail / 1000.0);
    hPsd->Fill(qTotal / 1000.0, psdValue[index]);
}

auto cPsd = new TCanvas("cPsd", "PSD maps", 1000, 430);
cPsd->Divide(2, 1);
cPsd->cd(1);
gPad->SetLogz();
hTailTotal->Draw("colz");
cPsd->cd(2);
gPad->SetLogz();
hPsd->Draw("colz");
cPsd->Draw();
cPsd->SaveAs("psd_maps.png");


<img src="psd_maps.png" alt="tail-total and PSD-total correlations" style="max-width:76%;" />


### 固定 $Q_{\rm total}$ 区间计算 FoM

示例选择 $40000\le Q_{\rm total}<60000$。先观察投影确定两个 peak 的拟合范围，再分别进行 Gaussian fit。这里低 PSD peak 对应 gamma，高 PSD peak 对应 neutron。GetParameter(1) 和 GetParameter(2) 分别取得 Gaussian 的 mean 和 $\sigma$。


<div class="pyroot-code-marker"></div>

~~~python
charge_low, charge_high = 40000.0, 60000.0
c_fom_region = ROOT.TCanvas("c_fom_region_py", "selected interval", 760, 500)
c_fom_region.SetLogz()
h_psd.Draw("colz")
line_charge_low = ROOT.TLine(charge_low / 1000.0, 0.0, charge_low / 1000.0, 0.5)
line_charge_high = ROOT.TLine(charge_high / 1000.0, 0.0, charge_high / 1000.0, 0.5)
for line in (line_charge_low, line_charge_high):
    line.SetLineColor(ROOT.kRed + 1)
    line.SetLineWidth(2)
    line.Draw()
c_fom_region.Draw()
c_fom_region.SaveAs("fom_region.png")

h_slice = ROOT.TH1D(
    "h_slice_py",
    "40000 #leq Q_{total} < 60000;Q_{tail}/Q_{total};counts",
    160, 0.0, 0.4
)
for charge, psd in zip(total_charge, psd_value):
    if charge_low <= charge < charge_high and math.isfinite(psd):
        h_slice.Fill(psd)

fit_gamma = ROOT.TF1("fit_gamma_py", "gaus", 0.04, 0.12)
fit_neutron = ROOT.TF1("fit_neutron_py", "gaus", 0.19, 0.34)
fit_gamma.SetParameters(h_slice.GetMaximum(), 0.074, 0.009)
fit_neutron.SetParameters(60.0, 0.27, 0.019)
result_gamma = h_slice.Fit(fit_gamma, "LRSQ0")
result_neutron = h_slice.Fit(fit_neutron, "LRSQ0")

mean_gamma = fit_gamma.GetParameter(1)
mean_neutron = fit_neutron.GetParameter(1)
fwhm_gamma = 2.355 * abs(fit_gamma.GetParameter(2))
fwhm_neutron = 2.355 * abs(fit_neutron.GetParameter(2))
fom = abs(mean_neutron - mean_gamma) / (fwhm_neutron + fwhm_gamma)

fit_gamma.SetLineColor(ROOT.kBlue + 1)
fit_neutron.SetLineColor(ROOT.kRed + 1)
c_fom = ROOT.TCanvas("c_fom_py", "FoM", 760, 500)
h_slice.Draw("E")
fit_gamma.Draw("same")
fit_neutron.Draw("same")
legend_fom = ROOT.TLegend(0.62, 0.68, 0.87, 0.85)
legend_fom.AddEntry(fit_gamma, "gamma", "l")
legend_fom.AddEntry(fit_neutron, "neutron", "l")
legend_fom.Draw()
label_fom = ROOT.TLatex()
label_fom.SetNDC()
label_fom.SetTextSize(0.04)
label_fom.DrawLatex(0.16, 0.80, f"FoM = {fom:.2f}")
c_fom.Draw()
c_fom.SaveAs("fom_projection.png")

print(f"gamma: mu={mean_gamma:.4f}, FWHM={fwhm_gamma:.4f}")
print(f"neutron: mu={mean_neutron:.4f}, FWHM={fwhm_neutron:.4f}")
print(f"FoM = {fom:.3f}; fit status = {int(result_gamma)}/{int(result_neutron)}")
~~~


In [ ]:
const double chargeLow = 40000.0;
const double chargeHigh = 60000.0;
auto cFomRegion = new TCanvas("cFomRegion", "selected interval", 760, 500);
cFomRegion->SetLogz();
hPsd->Draw("colz");
auto lineChargeLow = new TLine(chargeLow / 1000.0, 0.0, chargeLow / 1000.0, 0.5);
auto lineChargeHigh = new TLine(chargeHigh / 1000.0, 0.0, chargeHigh / 1000.0, 0.5);
for (auto line : {lineChargeLow, lineChargeHigh}) {
    line->SetLineColor(kRed + 1);
    line->SetLineWidth(2);
    line->Draw();
}
cFomRegion->Draw();
cFomRegion->SaveAs("fom_region.png");

auto hSlice = new TH1D(
    "hSlice", "40000 #leq Q_{total} < 60000;Q_{tail}/Q_{total};counts",
    160, 0.0, 0.4);
for (int index = 0; index < pulseCount; ++index) {
    if (totalCharge[index] >= chargeLow && totalCharge[index] < chargeHigh
        && std::isfinite(psdValue[index])) {
        hSlice->Fill(psdValue[index]);
    }
}

auto fitGamma = new TF1("fitGamma", "gaus", 0.04, 0.12);
auto fitNeutron = new TF1("fitNeutron", "gaus", 0.19, 0.34);
fitGamma->SetParameters(hSlice->GetMaximum(), 0.074, 0.009);
fitNeutron->SetParameters(60.0, 0.27, 0.019);
TFitResultPtr resultGamma = hSlice->Fit(fitGamma, "LRSQ0");
TFitResultPtr resultNeutron = hSlice->Fit(fitNeutron, "LRSQ0");

const double meanGamma = fitGamma->GetParameter(1);
const double meanNeutron = fitNeutron->GetParameter(1);
const double fwhmGamma = 2.355 * std::abs(fitGamma->GetParameter(2));
const double fwhmNeutron = 2.355 * std::abs(fitNeutron->GetParameter(2));
const double fom = std::abs(meanNeutron - meanGamma)
    / (fwhmNeutron + fwhmGamma);

fitGamma->SetLineColor(kBlue + 1);
fitNeutron->SetLineColor(kRed + 1);
auto cFom = new TCanvas("cFom", "FoM", 760, 500);
hSlice->Draw("E");
fitGamma->Draw("same");
fitNeutron->Draw("same");
auto legendFom = new TLegend(0.62, 0.68, 0.87, 0.85);
legendFom->AddEntry(fitGamma, "gamma", "l");
legendFom->AddEntry(fitNeutron, "neutron", "l");
legendFom->Draw();
auto labelFom = new TLatex();
labelFom->SetNDC();
labelFom->SetTextSize(0.04);
labelFom->DrawLatex(0.16, 0.80, Form("FoM = %.2f", fom));
cFom->Draw();
cFom->SaveAs("fom_projection.png");

std::cout << std::fixed << std::setprecision(4)
          << "gamma: mu=" << meanGamma << ", FWHM=" << fwhmGamma << "\n"
          << "neutron: mu=" << meanNeutron << ", FWHM=" << fwhmNeutron << "\n"
          << std::setprecision(3) << "FoM = " << fom << "; fit status = "
          << static_cast<int>(resultGamma) << "/"
          << static_cast<int>(resultNeutron) << std::endl;


<div style="display:flex; gap:1rem; align-items:flex-start;">
  <img src="fom_region.png" alt="selected Q total interval" style="width:47%;" />
  <img src="fom_projection.png" alt="PSD projection and Gaussian fits" style="width:47%;" />
</div>

该示例得到 $FoM\approx2.98$。这个数值只对应所选 $Q_{\rm total}$ 区间和积分门；不能直接当作整个二维分布的单一性能指标。


### 平均脉冲与积分门

用两个 fitted means 的中点作临时分界，将同一 $Q_{\rm total}$ 区间内的波形按 $Q_{\rm total}$ 归一后分别平均。下图用于观察慢成分差异，并帮助判断 $t_1$、$t_2$ 是否覆盖了有用的波形区间。


<div class="pyroot-code-marker"></div>

~~~python
split_psd = 0.5 * (mean_gamma + mean_neutron)
average_gamma = [0.0] * sample_count
average_neutron = [0.0] * sample_count
number_gamma = number_neutron = 0

for values, charge, psd in zip(aligned_pulses, total_charge, psd_value):
    if not charge_low <= charge < charge_high:
        continue
    target = average_gamma if psd < split_psd else average_neutron
    for i, value in enumerate(values):
        target[i] += value / charge
    if psd < split_psd:
        number_gamma += 1
    else:
        number_neutron += 1

average_gamma = [value / number_gamma for value in average_gamma]
average_neutron = [value / number_neutron for value in average_neutron]
difference = [n - g for n, g in zip(average_neutron, average_gamma)]
g_average_gamma = ROOT.TGraph(sample_count, samples, array("d", average_gamma))
g_average_neutron = ROOT.TGraph(sample_count, samples, array("d", average_neutron))
g_difference = ROOT.TGraph(sample_count, samples, array("d", difference))
g_average_gamma.SetLineColor(ROOT.kBlue + 1)
g_average_neutron.SetLineColor(ROOT.kRed + 1)
g_difference.SetLineColor(ROOT.kBlack)

c_average = ROOT.TCanvas("c_average_py", "average pulses", 800, 650)
c_average.Divide(1, 2)
c_average.cd(1)
average_graphs = ROOT.TMultiGraph()
average_graphs.SetTitle("Normalized average pulses;sample index;average amplitude")
average_graphs.Add(g_average_gamma)
average_graphs.Add(g_average_neutron)
average_graphs.Draw("AL")
legend_average = ROOT.TLegend(0.68, 0.68, 0.87, 0.84)
legend_average.AddEntry(g_average_gamma, "gamma", "l")
legend_average.AddEntry(g_average_neutron, "neutron", "l")
legend_average.Draw()
c_average.cd(2)
g_difference.SetTitle("neutron - gamma;sample index;amplitude difference")
g_difference.Draw("AL")
zero_difference = ROOT.TLine(0.0, 0.0, sample_count - 1.0, 0.0)
zero_difference.SetLineStyle(2)
zero_difference.Draw()
c_average.Draw()
c_average.SaveAs("average_pulses.png")
print(f"averaged {number_gamma} gamma-like and {number_neutron} neutron-like pulses")
~~~


In [ ]:
const double splitPsd = 0.5 * (meanGamma + meanNeutron);
std::vector<double> averageGamma(sampleCount, 0.0);
std::vector<double> averageNeutron(sampleCount, 0.0);
int numberGamma = 0;
int numberNeutron = 0;
for (int index = 0; index < pulseCount; ++index) {
    const double charge = totalCharge[index];
    if (charge < chargeLow || charge >= chargeHigh) continue;
    auto& average = psdValue[index] < splitPsd ? averageGamma : averageNeutron;
    for (int i = 0; i < sampleCount; ++i) {
        average[i] += alignedPulses[index][i] / charge;
    }
    if (psdValue[index] < splitPsd) ++numberGamma;
    else ++numberNeutron;
}
std::vector<double> difference(sampleCount);
for (int i = 0; i < sampleCount; ++i) {
    averageGamma[i] /= numberGamma;
    averageNeutron[i] /= numberNeutron;
    difference[i] = averageNeutron[i] - averageGamma[i];
}

auto gAverageGamma = new TGraph(sampleCount, samples.data(), averageGamma.data());
auto gAverageNeutron = new TGraph(sampleCount, samples.data(), averageNeutron.data());
auto gDifference = new TGraph(sampleCount, samples.data(), difference.data());
gAverageGamma->SetLineColor(kBlue + 1);
gAverageNeutron->SetLineColor(kRed + 1);
gDifference->SetLineColor(kBlack);

auto cAverage = new TCanvas("cAverage", "average pulses", 800, 650);
cAverage->Divide(1, 2);
cAverage->cd(1);
auto averageGraphs = new TMultiGraph();
averageGraphs->SetTitle("Normalized average pulses;sample index;average amplitude");
averageGraphs->Add(gAverageGamma);
averageGraphs->Add(gAverageNeutron);
averageGraphs->Draw("AL");
auto legendAverage = new TLegend(0.68, 0.68, 0.87, 0.84);
legendAverage->AddEntry(gAverageGamma, "gamma", "l");
legendAverage->AddEntry(gAverageNeutron, "neutron", "l");
legendAverage->Draw();
cAverage->cd(2);
gDifference->SetTitle("neutron - gamma;sample index;amplitude difference");
gDifference->Draw("AL");
auto zeroDifference = new TLine(0.0, 0.0, sampleCount - 1.0, 0.0);
zeroDifference->SetLineStyle(2);
zeroDifference->Draw();
cAverage->Draw();
cAverage->SaveAs("average_pulses.png");
std::cout << "averaged " << numberGamma << " gamma-like and "
          << numberNeutron << " neutron-like pulses" << std::endl;


<img src="average_pulses.png" alt="normalized average gamma and neutron pulses" style="max-width:58%;" />

平均 pulse maximum 已对齐，主要差别出现在 peak 之后的衰减区，因此 fast gate 的右边界应位于两类波形差异开始积累的位置附近，total gate 则应继续覆盖慢成分。最终边界仍由 FoM 比较确定。


## 作业要求

1. 观察若干原始波形，确定基线区间，并比较基线修正和时间对齐前后的波形。
2. 处理全部 10 000 条波形，绘制 $Q_{\rm tail}$–$Q_{\rm total}$ 和 $PSD$–$Q_{\rm total}$ 二维关联图，辨认 gamma 与 neutron 两条带。
3. 在两条带均有足够事例的一个 $Q_{\rm total}$ 区间内投影 PSD，拟合两个 peak 并计算 FoM。
4. 根据平均脉冲形状选择合理的 $t_0$ 和 $t_2$，再比较若干 $t_1$；用 FoM 给出所选 fast gate 和 total gate，同时以采样点和 ns 报告范围。
5. 用当前 PSD 结果暂时选择 gamma 与 neutron 事例，分别作总面积归一后求平均脉冲。比较两条平均波形及其差值，说明积分门为何能区分两类事例。这里的分类来自同一 PSD 变量，只用于解释波形差异，不作为独立验证。

进度对应第 5 章。
